In [33]:
import os
print(os.getcwd())
print(os.listdir())

/drive/notebooks/cind-820-learning-analytics/notebooks
['oulad_eda.ipynb', 'oulad_modelling.ipynb']


In [34]:
!mamba install pandas matplotlib numpy seaborn scikit-learn 

mambajs 0.21.1

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas, seaborn, scikit-learn
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 0.7597000000029802 seconds
All requested packages already installed.


In [35]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

raw_data_path = PROJECT_ROOT / "data" / "raw"
processed_data_path = PROJECT_ROOT / "data" / "processed"
fig_path = PROJECT_ROOT / "reports" / "figures"

print("Current directory:", NOTEBOOK_DIR)
print("Project root:", PROJECT_ROOT)
print("Raw data exists:", raw_data_path.exists())
print("Processed data exists:", processed_data_path.exists())
print("Figures folder exists:", fig_path.exists())


Current directory: /drive/notebooks/cind-820-learning-analytics/notebooks
Project root: /drive/notebooks/cind-820-learning-analytics
Raw data exists: True
Processed data exists: True
Figures folder exists: True


In [36]:
import os
print(os.getcwd())
print(os.listdir())

/drive/notebooks/cind-820-learning-analytics/notebooks
['oulad_eda.ipynb', 'oulad_modelling.ipynb']


In [37]:

df = pd.read_csv(processed_data_path / "oulad_student_level.csv")

print(df.shape)
df.head()

(32593, 15)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,mean_score,total_assessments,mean_weight
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,82.0,5.0,20.0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,66.4,5.0,20.0
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn,NaN,NaN,NaN
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass,76.0,5.0,20.0
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass,54.4,5.0,20.0


In [38]:
df["at_risk"] = df["final_result"].isin(["Fail", "Withdrawn"]).astype(int)

print(df["at_risk"].value_counts())

at_risk
1    17208
0    15385
Name: count, dtype: int64


In [39]:
feature_cols = [
    "code_module",
    "code_presentation",
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "disability",
    "num_of_prev_attempts",
    "studied_credits",
    "mean_score",
    "total_assessments",
    "mean_weight"
]

X = df[feature_cols]
y = df["at_risk"]

In [40]:
numeric_features = [
    "num_of_prev_attempts",
    "studied_credits",
    "mean_score",
    "total_assessments",
    "mean_weight"
]

categorical_features = [
    "code_module",
    "code_presentation",
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "disability"
]

In [41]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

In [42]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(26074, 13) (6519, 13)
at_risk
1    0.527959
0    0.472041
Name: proportion, dtype: float64
at_risk
1    0.527995
0    0.472005
Name: proportion, dtype: float64


In [43]:
# Logistic Regression
log_reg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

log_reg_pipeline.fit(X_train, y_train)
y_pred_log = log_reg_pipeline.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_log))
print(classification_report(y_test, y_pred_log))

Logistic Regression Accuracy: 0.9038196042337782
              precision    recall  f1-score   support

           0       0.88      0.93      0.90      3077
           1       0.93      0.88      0.91      3442

    accuracy                           0.90      6519
   macro avg       0.90      0.91      0.90      6519
weighted avg       0.91      0.90      0.90      6519



In [27]:
tree_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(random_state=42, max_depth=5))
])

tree_pipeline.fit(X_train, y_train)
y_pred_tree = tree_pipeline.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_tree))
print(classification_report(y_test, y_pred_tree))

Decision Tree Accuracy: 0.9292836324589661
              precision    recall  f1-score   support

           0       0.88      0.98      0.93      3077
           1       0.98      0.89      0.93      3442

    accuracy                           0.93      6519
   macro avg       0.93      0.93      0.93      6519
weighted avg       0.93      0.93      0.93      6519



In [44]:
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    ))
])

rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

Random Forest Accuracy: 0.9337321675103544
              precision    recall  f1-score   support

           0       0.90      0.97      0.93      3077
           1       0.97      0.90      0.94      3442

    accuracy                           0.93      6519
   macro avg       0.93      0.94      0.93      6519
weighted avg       0.94      0.93      0.93      6519



In [45]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_log),
        accuracy_score(y_test, y_pred_tree),
        accuracy_score(y_test, y_pred_rf)
    ]
})

results = results.sort_values("Accuracy", ascending=False)
display(results)

results.to_csv(processed_data_path / "model_results.csv", index=False)

,Model,Accuracy
2,Random Forest,0.933732
1,Decision Tree,0.929284
0,Logistic Regression,0.903820


In [46]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_cv_scores = cross_val_score(
    rf_pipeline,
    X,
    y,
    cv=cv,
    scoring="accuracy"
)

print("Random Forest CV scores:", rf_cv_scores)
print("Mean CV accuracy:", rf_cv_scores.mean())

Random Forest CV scores: [0.93403896 0.93680012 0.93680012 0.94216017 0.93003989]
Mean CV accuracy: 0.9359678539672149


In [48]:
results.to_csv(processed_data_path / "model_results.csv", index=False)

In [47]:
cv_results = pd.DataFrame({
    "fold": [1, 2, 3, 4, 5],
    "rf_accuracy": rf_cv_scores
})

cv_results.to_csv(processed_data_path / "rf_cv_scores.csv", index=False)
cv_results

,fold,rf_accuracy
0,1,0.934039
1,2,0.936800
2,3,0.936800
3,4,0.942160
4,5,0.930040
